Training a MLP with the Gabor Transform
===

# EEG-Audio Matching with Gabor-Transformed MLP

## Project Overview

This notebook implements Multi-Layer Perceptron models for matching EEG brain signals with corresponding audio signals using **Gabor-transformed data**. Unlike the raw signal approach, this uses time-frequency representations for richer feature extraction.

**What is the Gabor Transform?**
The Gabor transform is a mathematical operation that converts time-domain signals into a time-frequency representation. It's like creating a spectrogram that shows both when and at what frequency energy appears in a signal.

- **Time domain (raw):** Signal values over time, e.g., [0.1, -0.3, 0.5, ...]
- **Gabor transform:** 2D matrix showing energy at different times and frequencies
- **Dimensions:** M=16 (time bins) × N=64 (frequency bins) = 1,024 coefficients per channel

**Why Use Gabor Transform?**
1. **Captures frequency patterns:** Brain and audio signals have important frequency components
2. **Reduces dimensionality:** 320 timesteps → 16 time bins (20x compression)
3. **Better features:** Time-frequency patterns are more informative than raw values
4. **Neural efficiency:** 2D convolutions can process structured time-frequency data

**Dataset:**
- EEG: 64 channels × 16 time × 64 freq = 65,536 coefficients (Gabor-transformed)
- Audio: 1 channel × 16 time × 64 freq = 1,024 coefficients (Gabor-transformed)
- Total input: 66,560 features (same as raw, but now in time-frequency domain)

**Task:** Binary classification - determine if an EEG-audio pair matches (1) or not (0)

**Comparison to Raw Signals:**
- **Raw approach:** 320×64=20,480 EEG + 320×5=1,600 audio = 22,080 features (time-domain)
- **Gabor approach:** 64×16×64=65,536 EEG + 16×64=1,024 audio = 66,560 features (time-frequency)
- Gabor has more features but they're better structured for pattern recognition!

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.nn.functional import relu, sigmoid, tanh
import matplotlib.pyplot as plt
from tqdm import tqdm
from gabor import GaborDataset
import scipy.io as sio
import numpy as np

## Data Loading and Preprocessing

The dataset uses Gabor-transformed EEG and audio signals with a 5x expansion strategy.

**Gabor Transform Parameters:**
- **M = 16:** Number of time bins (temporal resolution)
- **N = 64:** Number of frequency bins (spectral resolution)
- **DGT:** Discrete Gabor Transform applied to convert time-domain → time-frequency domain

**Dataset Expansion:**
- Original: 1,160 EEG clips (each with 5 audio candidates)
- Expanded: 5,800 samples (1,160 × 5)
- Each EEG clip is paired with all 5 audio candidates separately

**Why 5x Expansion?**
This creates a balanced dataset where the model sees both:
- **Positive samples:** Correct EEG-audio matches (label = 1)
- **Negative samples:** Incorrect EEG-audio mismatches (label = 0)

For each EEG, 1 out of 5 pairs has label=1, and 4 have label=0.

**Data Shapes:**
- EEG input: (batch, 64, 16×64) = (batch, 64, 1024) after flattening
- Audio input: (batch, 16×64) = (batch, 1024) after flattening
- Combined: (batch, 66560) - concatenated features for MLP input

## Loading Data

In [ ]:
# Loading data handled by PyTorch dataloader, implemented in 'clips_uniform.py'
training_data = GaborDataset(mat_file='../gabor_results.mat', train=True)
testing_data = GaborDataset(mat_file='../gabor_results.mat', train=False)

train_dataloader = DataLoader(training_data, batch_size=4, shuffle=True)
test_dataloader = DataLoader(testing_data, batch_size=4, shuffle=True)

### Visualization

In [ ]:
for X, y in train_dataloader:
    print(f"Shape of batch X:\t{len(X)}")

    eeg = X[0]
    audio = X[1]
    print(f"Shape of EEG:\t{eeg.shape}")
    print(f"Shape of Audio:\t{audio.shape}")

    # torch.t - Transpose
    # Plot EEG channel 9 from the first set of features of the batch
    plt.plot(torch.t(eeg[0])[:,9])
    break


## Neural Network Architecture

We implement a multilayer perceptron (MLP) with the following architecture:

$$
\begin{align*}
  \mathbf H &= \operatorname{ReLU}(\mathbf X \mathbf W^{(1)} + \mathbf b^{(1)}) \\
  \mathbf O &= \operatorname{softmax}(\mathbf H \mathbf W^{(2)} + \mathbf b^{(2)}).
\end{align*}
$$

The matrix $\mathbf X \in \mathbb R^{M \times D}$ above corresponds to a minibatch of $M$ datapoints $\mathbf x_i \in \mathbb R^D$. The matrix $\mathbf H \in \mathbb R^{M \times J}$ is the hidden layer. For this problem, we choose the width of the hidden layer to be $J = 256$. The $i$-th row of the output matrix $\mathbf O \in \mathbb R^{M \times K}$ corresponds to the prediction $\hat{\mathbf y} \in \mathbb R^K$ of the $i$-th datapoint. We one hot encode the labels and use a softmax on the last layer so that our MLP models a proper distribution $p(y_n|\mathbf x_n)$. Given an image $\mathbf x^*$, if the $k$-th entry of our output $\hat{\mathbf y}^*$ is the largest, then we predict class $k$. This corresponds to picking the class $y^*$ that maximizes $p(y^* | \mathbf x)$.

In [ ]:
"""
We are given signals of dimension 320 by 69 (64 EEG plus 5 Audio), and we want to classify them as one of 5 classes.
At each step, we will be applying a linear transformation (W's) followed by an additive bias term (b's).
We then apply a non-linearity (relu or softmax).
"""

n_inputs = 66560 #D
n_hiddens = 512 #J
n_outputs = 1 #K

W1 = torch.nn.Parameter(torch.randn(size = (n_inputs, n_hiddens))) # matrix applying linear transformation
b1 = torch.nn.Parameter(torch.zeros(size = (1, n_hiddens))) # additive bias term
W2 = torch.nn.Parameter(torch.randn(size = (n_hiddens, n_outputs))) # matrix applying linear transformation
b2 = torch.nn.Parameter(torch.zeros(size = (1, n_outputs))) # additive bias term

params = [W1, b1, W2, b2]

In [ ]:
def net(X):
    """
    Implements the forward pass of our neural network. 
    
    :param X: a batch of signals of shape (batch_size, 320, 69).
    :return: a 2D numpy array of shape (batch_size, 5) where each row is a probability vector of length 5.
    """

    # Transpose and flatten EEG signals
    eeg = torch.transpose(X[0], dim0=1, dim1=2)
    eeg = torch.flatten(eeg, start_dim=1)

    # Transpose and flatten audio signals
    # audio = torch.transpose(X[1], dim0=1, dim1=2)
    audio = torch.flatten(X[1], start_dim=1)

    # Concatenate signals
    X = torch.cat((eeg, audio), dim=1)
    # Parameters are float32
    X = X.to(torch.float32)
    H = tanh(X @ W1 + b1)
    O = sigmoid(H @ W2 + b2)
    return O

## Training Configuration

**Hyperparameters:**
- **Learning rate (lr):** 10 - Very aggressive! Same as raw signal approach
- **Batch size:** 120 - Moderate batch size for stable gradients  
- **Epochs:** 10 - Shorter training than raw (100 epochs)
  - Gabor features may converge faster than raw signals
- **Hidden units:** 512 - Same capacity as raw signal model

**Architecture:**
- **Input → Hidden:** Tanh activation
  - Input: 66,560 Gabor features (EEG + audio concatenated)
  - Hidden: 512 units
  - Activation: Tanh(XW₁ + b₁) - range [-1, 1]
  
- **Hidden → Output:** Sigmoid activation
  - Output: 1 unit (binary classification)
  - Activation: Sigmoid(HW₂ + b₂) - probability in [0, 1]

**Loss Function:** Binary Cross-Entropy
- Formula: `-1.2 * [y * log(p)] - [(1-y) * log(1-p)]`
- Note: Factor of 1.2 on positive samples gives more weight to true matches
- This addresses class imbalance (1 positive vs 4 negatives per EEG)

**Optimizer:** Stochastic Gradient Descent (manual implementation)
- Updates: `param = param - lr * gradient`
- Gradients computed automatically by PyTorch's autograd

**Expected Performance:**
Gabor-transformed features should outperform raw signals because:
1. Time-frequency patterns are more discriminative
2. Dimensionality reduction (320 → 16 time bins) reduces noise
3. Structured representation is easier for neural networks to learn

**Comparison to Raw Signals:**
- Raw: 22,080 inputs, 512 hidden, 100 epochs
- Gabor: 66,560 inputs, 512 hidden, 10 epochs
- Gabor has 3× more features but should converge faster!

In [ ]:
## Testing neural net
for X, _ in train_dataloader:
    print(X[1].shape)
    print(f"{X[0].shape}")
    print(f"{net(X).shape}")
    print(f"{net(X)}")
    break

## Training the Neural Network

We also need to be able to evaluate our model prediction $\hat {\mathbf y}$ with the true label $\mathbf y$. We can do this through with cross entropy loss. If $K$ is the number of categories we have, then $\hat {\mathbf y}$ and $\mathbf y$ are $K$-dimensional one hot vectors. With this representation, cross entropy loss is defined as $$\ell(\hat {\mathbf y}, \mathbf y) = -\sum_{k=1}^K \mathbf y_i \log \hat {\mathbf y}_i.$$ Note that the way we structured our neural network, only $\hat {\mathbf y}$ is one hot encoded, and the labels $y$ coming from the MNIST training set are integers. This means we can much more simply write $$\ell(\hat {\mathbf y}, y) = -\log \hat {\mathbf y}_y.$$

For any general loss function, we can implement the gradient descent update step: $$w \leftarrow w - \eta \, \nabla_w \mathcal L$$ for every model parameter $w$ where $\mathcal L$ is the average loss over the minibatch.

In [ ]:
def bin_cross_entropy(y_hat, y):
    """
    Implement cross entropy loss for two inputs
    :param y_hat: a 2D numpy array of shape (batch_size,) where each entry is a probability of match.
    :param y: a 1D numpy array of shape (batch_size) where each element is a label (0 or 1).
  
    :return: a 1D numpy array of shape (batch_size) representing the cross entropy losses.
    """
    # Avoid nan values
    epsilon = 1e-7
    y_clamped = torch.clamp(y_hat, epsilon, 1 - epsilon)
    loss = -1 * 1.2 * y * torch.log(y_clamped) - (1 - y) * torch.log(1 - y_clamped)
    return loss


def sgd(params, lr=0.1):
    """
    Implements stochastic gradient descent. For each param, subtract the gradient times the learning rate.
    The gradient for each param is stored in param.grad. After updating each param, set its gradient to zero.
    
    :param params: a list of parameters to update.
    :param lr: the learning rate.
    
    :return: None
    """
    with torch.no_grad():
      for param in params:
        #print(param.grad)
        param -= lr*param.grad
        param.grad.zero_()

In [ ]:
torch.log(torch.tensor(1 - 1e-9))

In [ ]:
epochs = 10
batch_size = 120
lr = 10
train_iter = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_iter = DataLoader(testing_data, batch_size=batch_size, shuffle=True)


def accuracy(y_hat, y):
  with torch.no_grad():
    y_labels = torch.round(y_hat)
    correct = y_labels == y
    return correct.sum() / correct.numel()


def train(net, params, train_iter, test_iter, loss, updater):
  train_losses, train_accs = [], []
  test_losses, test_accs = [], []
  
  for epoch in tqdm(range(epochs)):
    train_loss, train_acc = 0.0, 0.0
    trials = 0

    for X, y in train_iter:
      trials += 1
      y_hat = net(X)
      l = loss(y_hat, y).mean()
      acc = accuracy(y_hat, y)

      l.backward()
      updater(params, lr)

      train_loss += l
      train_acc += acc

    train_losses.append(train_loss.item() / trials)
    train_accs.append(train_acc.item() / trials)

    test_loss, test_acc = 0.0, 0.0
    trials = 0

    y_pred = []
    y_true = []

    for X, y in test_iter:
      trials += 1
      with torch.no_grad():
        y_hat = net(X)

        # Recording for confusion matrix
        if epoch == epochs-1:
          y_pred += torch.round(y_hat)
          y_true += y
      
        l = loss(y_hat, y).mean()
        acc = accuracy(y_hat, y)

        test_loss += l
        test_acc += acc
    
    test_losses.append(test_loss.item() / trials)
    test_accs.append(test_acc.item() / trials)
  
  return train_losses, train_accs, test_losses, test_accs, y_pred, y_true

In [ ]:
"""
We are given signals of dimension 320 by 69 (64 EEG plus 5 Audio), and we want to classify them as one of 5 classes.
At each step, we will be applying a linear transformation (W's) followed by an additive bias term (b's).
We then apply a non-linearity (relu or softmax).
"""

n_inputs = 66560 #D
n_hiddens = 512 #J
n_outputs = 1 #K

W1 = torch.nn.Parameter(torch.randn(size = (n_inputs, n_hiddens))) # matrix applying linear transformation
b1 = torch.nn.Parameter(torch.zeros(size = (1, n_hiddens))) # additive bias term
W2 = torch.nn.Parameter(torch.randn(size = (n_hiddens, n_outputs))) # matrix applying linear transformation
b2 = torch.nn.Parameter(torch.zeros(size = (1, n_outputs))) # additive bias term

params = [W1, b1, W2, b2]

train_losses, train_accs, test_losses, test_accs, y_pred, y_true = train(net, params, train_iter, test_iter, loss=bin_cross_entropy, updater=sgd)

In [ ]:
#plt.plot(train_losses, c='crimson', label='train loss')
plt.plot(train_accs, '--', c='crimson', label='train acc')
#plt.plot(test_losses, c='navy', label='test loss')
plt.plot(test_accs, '--', c='navy', label='test acc')

plt.xlabel('epoch')
plt.grid()
plt.xlim(0, epochs-1)
plt.ylim(0, 1)
plt.legend()
#plt.savefig('LR=10')
plt.show()

In [ ]:
plt.plot(train_losses, c='crimson', label='train loss')
plt.plot(test_losses, c='navy', label='test loss')
plt.xlabel('epoch')
plt.grid()
plt.xlim(0, epochs-1)
plt.legend()
plt.show()

In [ ]:
print(y_pred)

In [ ]:
all_clips = sio.loadmat("../gabor_results.mat")['results']
results = all_clips['results'][0, 0]
ans = all_clips['ans'][0, 0][0]

In [ ]:
test_clips = [[torch.tensor(np.array(results[i,0:64])).unsqueeze(0),
                    torch.tensor(np.array(results[i,64+j])).unsqueeze(0)] for i in range(results.shape[0]) for j in range(5)]

In [ ]:
correct = []
for i in range(10):
    output = []
    for j in range(5):
        output.append(net(test_clips[i*5+j]))
    print(output)
    correct.append(output.index(max(output)) == ans[i])

print(correct)